# v2 — Notebook 09 · Hybrid RAG (lexical BM25 + semantic LSA)

**What RAG adds.** The query engine gives exact *numbers*; RAG retrieves *context* — country
descriptions, what the 2008/COVID/2022 events were, what a method means. Combined, an answer can say
both **what happened** (🔢 computed) and **why it might matter** (📚 retrieved · 🧠 inferred).

**Hybrid retrieval** = two retrievers, merged:
- **BM25** (lexical) — great at *exact terms* ("surplus", "Germany").
- **LSA** (semantic) — matches *meaning*, so "pandemic crash" finds the COVID doc even without the
  word "COVID".
- **Hybrid** — normalise both scores and combine, getting the strengths of each.

**Trust rule:** RAG supplies *context only*, never numbers. The KB contains no euro amounts; the
truth layer stays the only source of figures.
> Note: LSA is used for the semantic half because the installed Ollama models can't embed; the
> `search()` interface is unchanged if we later pull a neural embed model.

## Step 0 · Setup — build the retriever

In [1]:
import sys
from pathlib import Path
def find_v2_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "rag_system.py").exists():
            return base
    raise FileNotFoundError("v2 root not found")
V2 = find_v2_root(); sys.path.insert(0, str(V2))
DATA = V2.parent / "data" / "eu_trade_monthly.csv"

from src import preprocess as pp
from src import rag_system as rag
from src import interface as ui
from src import llm

kb = rag.knowledge_base()
hybrid = rag.HybridRetriever().fit(kb)
print(f"KB documents: {len(kb)}")
print("doc ids:", [i for i, _ in kb])

KB documents: 20
doc ids: ['DE', 'NL', 'FR', 'IT', 'PL', 'IE', 'exports', 'imports', 'balance', 'surplus', 'deficit', 'event_2008', 'event_covid', 'event_2022', 'event_rebound', 'stl', 'zscore', 'isoforest', 'sarima', 'seasonality']


## Step 1 · BM25 (lexical) — matches exact terms

**How to read:** scores are highest for docs literally containing the query words.

In [2]:
bm = rag.BM25().fit(kb)
for q in ["trade surplus", "isolation forest anomaly"]:
    print("Q:", q)
    for i, s in bm.search(q, k=3):
        print(f"   {s:5.2f}  {i}")
    print()

Q: trade surplus
    2.40  surplus
    2.20  balance
    1.90  NL

Q: isolation forest anomaly
    7.34  isoforest
    0.00  DE
    0.00  NL



## Step 2 · LSA (semantic) — matches meaning, not just words

**What:** query with words that *don't appear* in the target doc. **Why:** semantic retrieval finds
it anyway. **How to read:** "pandemic lockdown" surfaces the COVID doc though it says "COVID-19".

In [3]:
sem = rag.SemanticLSA().fit(kb)
for q in ["pandemic lockdown halted commerce", "war disrupted energy and trade"]:
    print("Q:", q)
    for i, s in sem.search(q, k=3):
        print(f"   {s:5.2f}  {i}")
    print()

Q: pandemic lockdown halted commerce
    0.97  event_covid
    0.00  surplus
    0.00  FR

Q: war disrupted energy and trade
    0.98  event_2022
    0.15  surplus
    0.15  NL



## Step 3 · Hybrid — lexical + semantic, side by side

**What:** compare BM25-only, semantic-only, and hybrid on the same query. **How to read:** hybrid
blends both rankings, so it is robust whether you use exact terms or paraphrase.

In [4]:
q = "which country sells the most abroad"
print("Q:", q, "\n")
print("BM25     :", [i for i, _ in bm.search(q, k=3)])
print("Semantic :", [i for i, _ in sem.search(q, k=3)])
print("Hybrid   :", [i for i, _ in hybrid.search(q, k=3)])

Q: which country sells the most abroad 

BM25     : ['exports', 'imports', 'surplus']
Semantic : ['imports', 'exports', 'surplus']
Hybrid   : ['exports', 'imports', 'surplus']


## Step 4 · The full trust-aware answer (the whole stack)

**What:** combine everything — the **query engine** computes the number, **RAG** adds context, the
**LLM** (if up) phrases it. **How to read:** 🔢 = computed truth · 📚 = retrieved context ·
🧠 = labelled inference · 💬 = LLM wording. The number only ever comes from 🔢.

In [5]:
# 1) exact fact from the query engine (the truth layer, via the interface)
df = pp.load_trade(DATA)
ans = ui.ask(df, "What was Spain's worst monthly export drop?")
# 2) context from RAG
ctx = hybrid.retrieve_texts(ans.computed + " export crash crisis", k=2)
# 3) assemble trust-aware answer
print(rag.explain(ans.computed, retrieved=ctx,
                  inferred="The timing matches a major global shock."))
# 4) optional LLM phrasing of the SAME number
phrased = llm.explain("Spain worst export drop", ans.computed,
                      inferred="timing matches a global shock")
print("\n💬", phrased if phrased else "(Ollama off — template/computed used)")

🔢 ES EXP: worst single-month fall was -35.0% in 2020-04-01.
📚 The COVID-19 pandemic in early 2020 triggered lockdowns that halted global trade; exports crashed about 30 percent in April 2020 before rebounding.
📚 The 2008 global financial crisis caused a sharp collapse in world trade in late 2008 and early 2009, with exports falling around 25-30 percent.
🧠 The timing matches a major global shock.

💬 During April 2020, Spain experienced its largest one-month decline in exports with a drop of -35.0%. This significant fall occurred against the backdrop of a global economic shock that had far-reaching impacts on trade worldwide.


## Step 5 · Summary

| Component | Role | Trust |
| --- | --- | --- |
| BM25 | lexical retrieval (exact terms) | 📚 context |
| LSA | semantic retrieval (meaning) | 📚 context |
| Hybrid | normalise + combine + re-rank | 📚 context |
| Query engine | the only source of numbers | 🔢 computed |
| LLM | rephrase computed facts | 💬 wording |

RAG enriches answers with *why/what* context while the **numbers stay exact and code-computed**.
Hybrid retrieval is robust to both keyword and paraphrased questions. Neural embeddings can replace
LSA later behind the same interface (pull `nomic-embed-text` in Ollama).

**→ Next: Notebook 10 · Evaluation & v1↔v2 comparison**, then the web app + write-up.